# 02 — Measuring conversations: the deterministic layer

## One number per handoff
For every pair of consecutive turns: **FTO = next.start_ms − prev.end_ms** (floor transfer offset — "the floor" = whose turn it is to speak).
- FTO **negative** → overlap → if >100ms, a **barge-in** (someone took the floor by force).
- FTO **positive** → gap → on user→agent handoffs this is **response latency**; >800ms = laggy.

Everything in the failure table is this subtraction plus two thresholds, and both thresholds live in `rubric.yaml`, not in code. You verified one by hand tonight (t2→t3 = −2,220ms on MUL0035).

## Why median and p90, never mean
Same culture as your perf work: latency distributions are skewed; a few terrible gaps drag a mean while the median stays honest, and the p90 tells you what the *bad tenth* of experiences feel like. Users do not experience averages; they experience tails.

## Stress profiles = workload classes
Each call gets a scenario-difficulty label (`clean`, `pause_heavy`, `interruption` — assigned by deterministic rules; `ambiguous`, `kb_gap` need semantics, so code never assigns them). Slicing results by class — the **cross-cut** — is how "the agent breaks specifically on hesitant callers" becomes visible. One blended score hides exactly that.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "rubric.yaml").exists())
import sys
sys.path.insert(0, str(ROOT / "pipeline"))
print("repo root:", ROOT)

import json
import numpy as np
import matplotlib.pyplot as plt
from signals import turn_metrics, analyze, load_rubric

rubric = load_rubric(ROOT / "rubric.yaml")
calls = {p.stem: json.loads(p.read_text()) for p in sorted((ROOT / "data" / "normalized").glob("*.json"))}
print(len(calls), "calls:", ", ".join(calls))

**PREDICT:** which call will have the worst (highest) median user→agent gap? You met `swz_MUL0056` tonight. Then run.

In [ ]:
rows = []
for cid, call in calls.items():
    r = analyze(call["turns"], rubric)
    rows.append((cid, call["stress_profile"], len(r["barge_ins"]), r["latency"]["n_laggy"],
                 r["latency"]["median_gap_ms"], r["latency"]["p90_gap_ms"]))
print(f"{'call':<14} {'profile':<13} {'barge':>5} {'laggy':>5} {'med':>7} {'p90':>7}")
for row in sorted(rows, key=lambda x: -(x[4] or 0)):
    print(f"{row[0]:<14} {row[1]:<13} {row[2]:>5} {row[3]:>5} {str(row[4]):>7} {str(row[5]):>7}")

## Mean vs median vs p90 — see it, not believe it
**PREDICT:** for MUL0056's gaps, will the mean be above or below the median, and why?

In [ ]:
gaps = [e["gap_ms"] for e in turn_metrics(calls["swz_MUL0056"]["turns"])
        if e["prev_spk"] == "user" and e["next_spk"] == "agent" and e["fto_ms"] >= 0]
gaps_np = np.array(gaps)
print(f"n={len(gaps)}  mean={gaps_np.mean():.0f}  median={np.median(gaps_np):.0f}  p90={np.percentile(gaps_np, 90):.0f}")
plt.figure(figsize=(8, 2.6))
plt.hist(gaps_np, bins=24, color="#7F77DD")
for v, lbl, c in [(gaps_np.mean(), "mean", "#D85A30"), (np.median(gaps_np), "median", "#1D9E75")]:
    plt.axvline(v, color=c, lw=2, label=lbl)
plt.legend(); plt.xlabel("user->agent gap (ms)"); plt.title("MUL0056 response gaps"); plt.show()

## Exercise — you own the threshold now
The rubric says laggy = >800ms. Product teams argue about this number constantly. **PREDICT how many laggy events the whole pool gains if the bar tightens to 500ms**, then run.

In [ ]:
def laggy_count(threshold_ms):
    total = 0
    for call in calls.values():
        hand = [e for e in turn_metrics(call["turns"])
                if e["prev_spk"] == "user" and e["next_spk"] == "agent" and e["fto_ms"] >= 0]
        total += sum(1 for e in hand if e["gap_ms"] > threshold_ms)
    return total

for th in (800, 500, 300):
    print(f"laggy events at >{th}ms: {laggy_count(th)}")

That sensitivity — "the failure count is a function of a config line" — is precisely why thresholds live in `rubric.yaml` and get disclosed, never buried. An engineer in the room may push on your 800; your answer: conversation-analysis baselines put natural handoffs ~200–300ms, sub-800 reads as acceptable assistant latency, and the rubric is one line to re-run under their number. Config, not dogma.

## Dirty data — the honesty section
Three real things tonight's corpus taught us, all worth saying out loud to engineers:
1. Some SpokenWOZ "overlaps" run 9+ seconds — ASR segmentation artifacts or crosstalk, not clean barge-ins. We *measure* faithfully but *select* mid-range exemplars (and say so).
2. Speaker truth came from annotation tags, not audio — diarization would have added noise we cannot audit.
3. If a source lacks `end_ms`, overlap is **uncomputable** — the pipeline refuses to fake it (latency-only treatment). Saying "we cannot know that from this data" is a credibility move, not a weakness.

## Self-check
1. Compute the FTO: prev.end = 30,000, next.start = 29,400. Event type?
2. Why is latency only computed on user→agent handoffs?
3. Your agent's mean latency improved 20% after a release but p90 doubled. What happened, in plain words?
4. What makes a stress profile different from a failure?

<details><summary>Answers</summary>

1. −600ms → overlap >100ms → barge-in (whoever spoke second interrupted).
2. We are scoring the agent's responsiveness; a human pausing before answering the bot is not a product defect. (Also: agent→user "gaps" are the human thinking — not ours to judge.)
3. Typical responses got a bit faster but the worst tenth got much worse — e.g. a cache that usually hits but stalls badly on miss. Means hide tail regressions.
4. Profile describes the input (scenario difficulty); failures describe the output (agent performance). Easy-input + failures = the most damning combination — pure product defect.
</details>